# Fase 2.5 — Blocos residuais no gerador (Colab GPU)

Troca o miolo do codificador: hoje ele é **denso** (cada bloco recebe a concatenação de todos
os anteriores, com a mensagem reinjetada a cada bloco); passa a ser **residual** (`x + conv(x)`),
como no STEGS2GAN. O skip global (`estego = portadora + resíduo`) permanece nos dois — ele
nunca foi o "bloco residual" do artigo.

**Por que é uma fase inteira e não um flag.** Mudar a topologia do gerador invalida o
`state_dict` da Fase 2: **não existe warm start**. Esta fase refaz o caminho 0 → 2 com o
gerador novo. A comparação com o denso é feita época a época, contra os CSVs que já estão
medidos — por isso não é preciso re-treinar o denso.

**Prazo.** Pode ser feita antes ou depois da Fase 3, mas **não depois da Fase 4**: dali em
diante existem medidas de segurança feitas sobre um gerador específico.

**Critérios de saída** (seção 8 avalia sozinha):
1. acurácia de bits ≥ 0,99;
2. PSNR ≥ 36,30 dB — não regredir o que a Fase 2 densa entregou;
3. custo por época medido e comparado;
4. os dois geradores preservados em disco.

**Ordem das seções:** 1 GPU · 2 Drive · 3 código · 4 dataset · 5 orçamento · 6 Fase 0 ·
7 Fase 2 · 8 comparação · 9 amostras · 10 antes de fechar.

## 1. Checar a GPU

In [ ]:
import torch, subprocess
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
else:
    print('ATENÇÃO: sem GPU. Runtime > Alterar tipo de runtime > GPU.')

## 2. Montar o Google Drive

Tudo que precisa sobreviver ao fim da sessão vai para `PASTA_BASE`. O ambiente do Colab é
efêmero: foi assim que os pesos das Fases 0 e 2 quase se perderam.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

PASTA_BASE   = '/content/drive/MyDrive/IC_stego'
PASTA_DADOS  = os.path.join(PASTA_BASE, 'datasets')
PASTA_RUNS   = os.path.join(PASTA_BASE, 'runs')
TRABALHO     = '/content/runs'

os.makedirs(PASTA_RUNS, exist_ok=True)
os.makedirs(TRABALHO, exist_ok=True)
print('base   :', PASTA_BASE)
print('dados  :', PASTA_DADOS)
print('espelho:', PASTA_RUNS)
print('\nJá existe no Drive:')
for d in sorted(os.listdir(PASTA_RUNS)):
    print('  ', d)

## 3. Gravar o código

Sem upload: as células abaixo escrevem os arquivos direto no ambiente. `fase25_residual.py`
é o único arquivo novo — ele troca o gerador usado por `phase0` e `phase2` sem alterar
nenhum dos dois.

In [ ]:
%%writefile fase0_codificador_decodificador.py
import os
import glob
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image


CONFIG = {
  
    "pasta_dados": None,
    "canais": 3,               # 1 = grayscale (BOSSBase) | 3 = RGB (DIV2K, COCO)
    "tamanho_img": 128,        
    "fracao_val": 0.05,       
    "max_imagens": None,      
    

    "profundidade_dados": 1,  
    "tamanho_msg": None,      


    "tamanho_lote": 16,       
    "epocas": 30,
    "taxa_aprend": 1e-3,      
    "largura": 32,             
    "lambda_imagem": 1.0,      
    "lambda_mensagem": 1.0,    
    "lambda_residuo": 0.0,    
    "usar_amp": True,          
    "alvo_bitacc": 0.99,       

    # --- infra ---
    "num_trabalhadores": 2,   
    "semente": 42,             
    "pasta_saida": "runs/phase0",
    "salvar_cada": 1,        
    "dispositivo": "cuda" if torch.cuda.is_available() else "cpu",
}



COLAB_GPU = dict(
    tamanho_img=128,
    tamanho_msg=32,         
    tamanho_lote=32,
    epocas=120,
    largura=64,
    max_imagens=None,        # dataset inteiro
    num_trabalhadores=4,
    lambda_imagem=2.0,
    lambda_residuo=0.05,
)


EXTENSOES_IMG = (".png", ".jpg", ".jpeg", ".bmp", ".pgm", ".ppm", ".tif", ".tiff", ".webp")


class DatasetImagens(Dataset):
    def __init__(self, caminhos, canais, tamanho_img, treino=True):
        self.caminhos = caminhos
        self.canais = canais
        self.modo = "L" if canais == 1 else "RGB"   # "L" = grayscale
        # Treino: recorte aleatório + espelhamento (aumenta os dados e desacopla do tamanho original).
        # Se a imagem for menor que o recorte, faz resize antes.
        if treino:
            self.transformacao = transforms.Compose([
                transforms.Resize(tamanho_img),                       # garante lado >= tamanho_img
                transforms.RandomCrop(tamanho_img, pad_if_needed=True),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),                                # pixels em [0, 1]
            ])
        else:
            self.transformacao = transforms.Compose([
                transforms.Resize(tamanho_img),
                transforms.CenterCrop(tamanho_img),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.caminhos)

    def __getitem__(self, indice):
        imagem = Image.open(self.caminhos[indice]).convert(self.modo)
        return self.transformacao(imagem)


class DatasetSintetico(Dataset):
    def __init__(self, canais, tamanho_img, n=512):
        self.canais, self.tamanho_img, self.n = canais, tamanho_img, n

    def __len__(self):
        return self.n

    def __getitem__(self, indice):
        # imagem suave (não ruído puro) para o MSE ter o que aprender
        base = torch.rand(self.canais, 1, 1)
        gradiente = torch.linspace(0, 1, self.tamanho_img).view(1, 1, -1)
        return (base + 0.3 * gradiente).clamp(0, 1).expand(
            self.canais, self.tamanho_img, self.tamanho_img).clone()


def montar_loaders(cfg):
    if cfg["pasta_dados"]:
        caminhos = []
        for ext in EXTENSOES_IMG:
            caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext), recursive=True)
            caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext.upper()), recursive=True)
        caminhos = sorted(set(caminhos))
        if not caminhos:
            raise FileNotFoundError(f"Nenhuma imagem encontrada em {cfg['pasta_dados']}")
        random.Random(cfg["semente"]).shuffle(caminhos)
        if cfg.get("max_imagens"):
            caminhos = caminhos[:cfg["max_imagens"]]
            print(f"[dados] max_imagens={cfg['max_imagens']} -> usando {len(caminhos)} imagens")
        n_val = max(1, int(len(caminhos) * cfg["fracao_val"]))
        caminhos_val, caminhos_treino = caminhos[:n_val], caminhos[n_val:]
        print(f"[dados] {len(caminhos_treino)} treino / {len(caminhos_val)} val  ({cfg['pasta_dados']})")
        ds_treino = DatasetImagens(caminhos_treino, cfg["canais"], cfg["tamanho_img"], treino=True)
        ds_val = DatasetImagens(caminhos_val, cfg["canais"], cfg["tamanho_img"], treino=False)
    else:
        print("[dados] pasta_dados=None -> usando dataset SINTÉTICO (teste rápido)")
        ds_treino = DatasetSintetico(cfg["canais"], cfg["tamanho_img"], n=512)
        ds_val = DatasetSintetico(cfg["canais"], cfg["tamanho_img"], n=64)

    loader_treino = DataLoader(ds_treino, batch_size=cfg["tamanho_lote"], shuffle=True,
                               num_workers=cfg["num_trabalhadores"], drop_last=True, pin_memory=True)
    loader_val = DataLoader(ds_val, batch_size=cfg["tamanho_lote"], shuffle=False,
                            num_workers=cfg["num_trabalhadores"], pin_memory=True)
    return loader_treino, loader_val


def bloco_conv(canais_entrada, canais_saida):
    return nn.Sequential(
        nn.Conv2d(canais_entrada, canais_saida, 3, padding=1),
        nn.BatchNorm2d(canais_saida),
        nn.LeakyReLU(0.2, inplace=True),
    )


class Codificador(nn.Module):

    def __init__(self, canais, profundidade_dados, largura=32):
        super().__init__()
        self.bloco1 = bloco_conv(canais, largura)
        self.bloco2 = bloco_conv(largura + profundidade_dados, largura)
        self.bloco3 = bloco_conv(2 * largura + profundidade_dados, largura)
        self.para_residuo = nn.Conv2d(3 * largura + profundidade_dados, canais, 3, padding=1)

    def forward(self, portadora, mensagem):
        a = self.bloco1(portadora)
        b = self.bloco2(torch.cat([a, mensagem], dim=1))
        c = self.bloco3(torch.cat([a, b, mensagem], dim=1))
        residuo = self.para_residuo(torch.cat([a, b, c, mensagem], dim=1))
        estego = (portadora + residuo).clamp(0.0, 1.0)   # soma o resíduo e mantém em [0,1]
        return estego


class Decodificador(nn.Module):
    

    def __init__(self, canais, profundidade_dados, largura=32):
        super().__init__()
        self.bloco1 = bloco_conv(canais, largura)
        self.bloco2 = bloco_conv(largura, largura)
        self.bloco3 = bloco_conv(2 * largura, largura)
        self.para_logits = nn.Conv2d(3 * largura, profundidade_dados, 3, padding=1)

    def forward(self, estego):
        a = self.bloco1(estego)
        b = self.bloco2(a)
        c = self.bloco3(torch.cat([a, b], dim=1))
        return self.para_logits(torch.cat([a, b, c], dim=1))


def _tamanho_msg(cfg):
    return cfg.get("tamanho_msg") or cfg["tamanho_img"]


def mensagem_aleatoria(tamanho_lote, cfg):
    tm = _tamanho_msg(cfg)
    return torch.randint(0, 2, (tamanho_lote, cfg["profundidade_dados"], tm, tm),
                         dtype=torch.float32, device=cfg["dispositivo"])


def espalhar_mensagem(mensagem, cfg):
    if _tamanho_msg(cfg) == cfg["tamanho_img"]:
        return mensagem
    return F.interpolate(mensagem, size=cfg["tamanho_img"], mode="nearest")


def agregar_logits(logits, cfg):
    if _tamanho_msg(cfg) == cfg["tamanho_img"]:
        return logits
    return F.adaptive_avg_pool2d(logits, _tamanho_msg(cfg))


@torch.no_grad()
def acuracia_bits(logits, mensagem):
    predito = (torch.sigmoid(logits) > 0.5).float()
    return (predito == mensagem).float().mean().item()


def _janela_gaussiana(tamanho, sigma, canais, dispositivo):
    coords = torch.arange(tamanho, device=dispositivo) - tamanho // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    janela2d = (g[:, None] * g[None, :])
    return janela2d.expand(canais, 1, tamanho, tamanho).contiguous()


@torch.no_grad()
def ssim(a, b, canais, tamanho_janela=11, sigma=1.5):

    janela = _janela_gaussiana(tamanho_janela, sigma, canais, a.device)
    pad = tamanho_janela // 2
    mu_a = F.conv2d(a, janela, padding=pad, groups=canais)
    mu_b = F.conv2d(b, janela, padding=pad, groups=canais)
    mu_a2, mu_b2, mu_ab = mu_a * mu_a, mu_b * mu_b, mu_a * mu_b
    var_a = F.conv2d(a * a, janela, padding=pad, groups=canais) - mu_a2
    var_b = F.conv2d(b * b, janela, padding=pad, groups=canais) - mu_b2
    cov_ab = F.conv2d(a * b, janela, padding=pad, groups=canais) - mu_ab
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    s = ((2 * mu_ab + c1) * (2 * cov_ab + c2)) / ((mu_a2 + mu_b2 + c1) * (var_a + var_b + c2))
    return s.mean().item()

def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = _tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2
    print(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"(tamanho_msg={tm}, tamanho_img={cfg['tamanho_img']})")

    loader_treino, loader_val = montar_loaders(cfg)

    codificador = Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)


    otimizador = torch.optim.Adam(
        list(codificador.parameters()) + list(decodificador.parameters()), lr=cfg["taxa_aprend"])
    usar_amp = cfg["usar_amp"] and dispositivo == "cuda"
    escalador = torch.cuda.amp.GradScaler(enabled=usar_amp)

    melhor = {"score": -1.0, "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0, "ssim": 0.0}

    for epoca in range(1, cfg["epocas"] + 1):
        codificador.train(); decodificador.train()
        acumulado = {"img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            mensagem = mensagem_aleatoria(portadora.size(0), cfg)
            mensagem_espalhada = espalhar_mensagem(mensagem, cfg)

            with torch.cuda.amp.autocast(enabled=usar_amp):
                estego = codificador(portadora, mensagem_espalhada)
                logits = agregar_logits(decodificador(estego), cfg)

                perda_imagem = F.mse_loss(estego, portadora)                              # qualidade visual
                perda_mensagem = F.binary_cross_entropy_with_logits(logits, mensagem)     # recuperação
                perda_residuo = (estego - portadora).abs().mean()                         # perturbação pequena
                perda = (cfg["lambda_imagem"] * perda_imagem
                         + cfg["lambda_mensagem"] * perda_mensagem
                         + cfg["lambda_residuo"] * perda_residuo)


            otimizador.zero_grad()
            escalador.scale(perda).backward()
            escalador.step(otimizador)
            escalador.update()

            n_lote = portadora.size(0)
            acumulado["img"] += perda_imagem.item() * n_lote
            acumulado["msg"] += perda_mensagem.item() * n_lote
            acumulado["acc"] += acuracia_bits(logits, mensagem) * n_lote
            acumulado["n"] += n_lote

        n = acumulado["n"]
        print(f"[treino] época {epoca:3d}  img_mse={acumulado['img']/n:.5f}  "
              f"msg_bce={acumulado['msg']/n:.4f}  acuracia_bits={acumulado['acc']/n:.4f}")

       
        val = avaliar(codificador, decodificador, loader_val, cfg)
        ok = "OK" if (val["acuracia_bits"] >= cfg["alvo_bitacc"] and val["psnr"] >= 35.0) else "  "
        print(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  linf={val['linf']:.4f}  {ok}")

      
        score = (val["psnr"] if val["acuracia_bits"] >= cfg["alvo_bitacc"]
                 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val},
                       os.path.join(cfg["pasta_saida"], "ckpt_best.pt"))

        if epoca % cfg["salvar_cada"] == 0:
            salvar_amostras(codificador, loader_val, cfg, epoca)
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "epoca": epoca, "cfg": cfg},
                       os.path.join(cfg["pasta_saida"], "ckpt_last.pt"))

    print(f"\n[fim] checkpoints e amostras em: {cfg['pasta_saida']}")
    print(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}")
    if melhor["acuracia_bits"] >= cfg["alvo_bitacc"] and melhor["psnr"] >= 35.0:
        print("[Fase 0 FECHADA] acurácia e PSNR>=35dB atingidos -> pode ligar a WGAN-GP (Fase 2).")
    else:
        print("[Fase 0 ainda aberta] baixar o payload (tamanho_msg menor) e/ou treinar mais antes da Fase 2.")
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"], "ssim": melhor["ssim"]}


@torch.no_grad()
def avaliar(codificador, decodificador, loader, cfg):
    codificador.eval(); decodificador.eval()
    acc, mse, ss, linf, n = 0.0, 0.0, 0.0, 0.0, 0
    for portadora in loader:
        portadora = portadora.to(cfg["dispositivo"])
        mensagem = mensagem_aleatoria(portadora.size(0), cfg)
        estego = codificador(portadora, espalhar_mensagem(mensagem, cfg))
        logits = agregar_logits(decodificador(estego), cfg)
        n_lote = portadora.size(0)
        acc += acuracia_bits(logits, mensagem) * n_lote
        mse += F.mse_loss(estego, portadora).item() * n_lote
        ss += ssim(estego, portadora, cfg["canais"]) * n_lote
        linf = max(linf, (estego - portadora).abs().max().item())
        n += n_lote
    mse = mse / max(n, 1)
    psnr = 10.0 * math.log10(1.0 / mse) if mse > 0 else float("inf")
    return {"acuracia_bits": acc / max(n, 1), "psnr": psnr, "ssim": ss / max(n, 1), "linf": linf}


@torch.no_grad()
def salvar_amostras(codificador, loader, cfg, epoca):
    codificador.eval()
    portadora = next(iter(loader)).to(cfg["dispositivo"])[:8]
    mensagem = mensagem_aleatoria(portadora.size(0), cfg)
    estego = codificador(portadora, espalhar_mensagem(mensagem, cfg))
    # amplia o resíduo p/ inspeção visual do que foi alterado
    residuo = (estego - portadora).abs()
    residuo = residuo / (residuo.amax() + 1e-8)
    grade = torch.cat([portadora, estego, residuo], dim=0)   # linha1=portadora, 2=estego, 3=resíduo
    caminho = os.path.join(cfg["pasta_saida"], f"samples_epoch{epoca:03d}.png")
    save_image(grade, caminho, nrow=8)


if __name__ == "__main__":
    treinar()


In [ ]:
%%writefile fase2_wgan_gp.py

import os
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import fase0_codificador_decodificador as base   # reusa arquitetura, loaders e métricas da Fase 0


CONFIG = dict(base.CONFIG)
CONFIG.update({
    "lambda_adv": 0.01,       
                               
    "n_critic": 5,             
    "lambda_gp": 10.0,         
    "largura_critico": 64,     
    "lr_critico": 1e-4,       
    "beta1": 0.0, "beta2": 0.9,  # betas do Adam (recomendado p/ WGAN-GP)

    "taxa_aprend": 1e-4,      
    "epocas": 60,
    "usar_amp": False,         # GP + AMP não combinam
    "carregar_fase0": "runs/phase0_bossbase/ckpt_best.pt",  # warm start; None = do zero
    "pasta_saida": "runs/phase2_bossbase",
})

COLAB_GPU = dict(base.COLAB_GPU)
COLAB_GPU.update({
    "epocas": 60,
    "lambda_adv": 0.01,
    "usar_amp": False,
})


class Critico(nn.Module):

    def __init__(self, canais, largura=64):
        super().__init__()

        def bloco(entrada, saida, norm=True):
            camadas = [nn.Conv2d(entrada, saida, 4, stride=2, padding=1)]  # reduz H,W pela metade
            if norm:
                camadas.append(nn.InstanceNorm2d(saida, affine=True))
            camadas.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*camadas)

        self.corpo = nn.Sequential(
            bloco(canais, largura, norm=False),   # 1ª camada sem norm
            bloco(largura, largura * 2),
            bloco(largura * 2, largura * 4),
            bloco(largura * 4, largura * 8),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)          # vira (B, C, 1, 1) p/ qualquer tamanho de entrada
        self.para_score = nn.Linear(largura * 8, 1)  # score escalar por imagem

    def forward(self, x):
        h = self.corpo(x)
        h = self.pool(h).flatten(1)     # (B, C)
        return self.para_score(h).squeeze(1)  # (B,)


def penalidade_gradiente(critico, reais, falsas, dispositivo):

    b = reais.size(0)
    eps = torch.rand(b, 1, 1, 1, device=dispositivo)
    interp = (eps * reais + (1.0 - eps) * falsas).requires_grad_(True)
    score = critico(interp)
    grads = torch.autograd.grad(
        outputs=score, inputs=interp,
        grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True, only_inputs=True,
    )[0]
    grads = grads.reshape(b, -1)
    return ((grads.norm(2, dim=1) - 1.0) ** 2).mean()


def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = base._tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2
    print(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"lambda_adv={cfg['lambda_adv']}  n_critic={cfg['n_critic']}")

    loader_treino, loader_val = base.montar_loaders(cfg)

    codificador = base.Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = base.Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    critico = Critico(cfg["canais"], cfg["largura_critico"]).to(dispositivo)

    # --- warm start da Fase 0 (colocar o adversário sobre uma base que já funciona) 
    ck = cfg.get("carregar_fase0")
    if ck and os.path.exists(ck):
        estado = torch.load(ck, map_location=dispositivo)
        codificador.load_state_dict(estado["codificador"])
        decodificador.load_state_dict(estado["decodificador"])
        print(f"[warm start] pesos da Fase 0 carregados de {ck}")
    else:
        print(f"[warm start] {ck!r} não encontrado -> treinando do zero "
              "(recomendado rodar a Fase 0 antes)")

    # gerador = codificador + decodificador (treinados juntos); crítico separado
    opt_g = torch.optim.Adam(list(codificador.parameters()) + list(decodificador.parameters()),
                             lr=cfg["taxa_aprend"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_c = torch.optim.Adam(critico.parameters(),
                             lr=cfg["lr_critico"], betas=(cfg["beta1"], cfg["beta2"]))

    melhor = {"score": -1.0, "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0, "ssim": 0.0}

    for epoca in range(1, cfg["epocas"] + 1):
        codificador.train(); decodificador.train(); critico.train()
        ac = {"c": 0.0, "w": 0.0, "adv": 0.0, "img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            b = portadora.size(0)

            # ============ 1) treina o CRÍTICO n_critic vezes ============
            for _ in range(cfg["n_critic"]):
                mensagem = base.mensagem_aleatoria(b, cfg)
                with torch.no_grad():                       # não propaga p/ o gerador aqui
                    estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
                score_real = critico(portadora).mean()
                score_falsa = critico(estego).mean()
                gp = penalidade_gradiente(critico, portadora, estego, dispositivo)
                # crítico MINIMIZA:  E[C(falsa)] - E[C(real)] + lambda_gp * GP
                perda_c = score_falsa - score_real + cfg["lambda_gp"] * gp
                opt_c.zero_grad(); perda_c.backward(); opt_c.step()

            w_dist = (score_real - score_falsa).item()      # estimativa da distância de Wasserstein

            # ============ 2) treina o GERADOR 1 vez ============
            mensagem = base.mensagem_aleatoria(b, cfg)
            estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
            logits = base.agregar_logits(decodificador(estego), cfg)

            perda_img = F.mse_loss(estego, portadora)
            perda_msg = F.binary_cross_entropy_with_logits(logits, mensagem)
            perda_res = (estego - portadora).abs().mean()
            perda_adv = -critico(estego).mean()             # gerador quer SUBIR o score da estego
            perda_g = (cfg["lambda_imagem"] * perda_img
                       + cfg["lambda_mensagem"] * perda_msg
                       + cfg["lambda_residuo"] * perda_res
                       + cfg["lambda_adv"] * perda_adv)
            opt_g.zero_grad(); perda_g.backward(); opt_g.step()

            ac["c"] += perda_c.item() * b;   ac["w"] += w_dist * b
            ac["adv"] += perda_adv.item() * b
            ac["img"] += perda_img.item() * b; ac["msg"] += perda_msg.item() * b
            ac["acc"] += base.acuracia_bits(logits, mensagem) * b
            ac["n"] += b

        n = ac["n"]
        print(f"[treino] época {epoca:3d}  perda_c={ac['c']/n:+.3f}  W={ac['w']/n:+.3f}  "
              f"adv={ac['adv']/n:+.3f}  img_mse={ac['img']/n:.5f}  "
              f"msg_bce={ac['msg']/n:.4f}  acuracia_bits={ac['acc']/n:.4f}")

        # --- validação (reusa a métrica da Fase 0) + amostras ---
        val = base.avaliar(codificador, decodificador, loader_val, cfg)
        ok = "OK" if (val["acuracia_bits"] >= cfg["alvo_bitacc"] and val["psnr"] >= 35.0) else "  "
        print(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  linf={val['linf']:.4f}  {ok}")

        score = (val["psnr"] if val["acuracia_bits"] >= cfg["alvo_bitacc"]
                 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val},
                       os.path.join(cfg["pasta_saida"], "ckpt_best.pt"))

        if epoca % cfg["salvar_cada"] == 0:
            base.salvar_amostras(codificador, loader_val, cfg, epoca)
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "epoca": epoca, "cfg": cfg},
                       os.path.join(cfg["pasta_saida"], "ckpt_last.pt"))

    print(f"\n[fim] checkpoints e amostras em: {cfg['pasta_saida']}")
    print(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}")
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"], "ssim": melhor["ssim"]}


if __name__ == "__main__":
    treinar()


In [ ]:
%%writefile logs_para_csv.py
#!/usr/bin/env python3
"""
Converte os logs de treino (.log, salvos do Colab) em CSVs de métricas.

Motivo: as Fases 0 e 2 e a varredura de lambda foram rodadas antes de o código passar
a gravar `metricas_treino.csv` / `metricas_val.csv`. O registro daquelas runs existe
só no texto do log. Este script recupera esses números para um formato tabular, sem
precisar re-treinar nada.

Uso:  python3 logs_para_csv.py            # varre os logs conhecidos e grava os CSVs
      python3 logs_para_csv.py --listar   # só mostra o que seria gerado

Normalização aplicada (os logs mudaram de nome entre versões do código):
    bit_acc     -> acuracia_bits
    W           -> w_dist
    psnr=34.5dB -> 34.5   (o sufixo "dB" é removido)
"""
import argparse
import csv
import json
import os
import re

RAIZ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))  # raiz do repositório

# Notebooks cuja SAÍDA das células é o único registro de um treino. Os .log abaixo são
# gerados a partir deles; se o .log já existir e for igual, nada muda.
# (notebook, log a gerar)
NOTEBOOKS = [
    ("notebooks/fase2_colab.ipynb", "runs/phase2_bossbase/train_colab.log"),
]

# (log de entrada, pasta de saída, rótulo do run)
FONTES = [
    ("runs/phase0_bossbase/train_colab.log", "runs/phase0_bossbase", "phase0_bossbase"),
    ("runs/logs_brutos/phase0_bossbase.log", "runs/phase0_local_cpu", "phase0_local_cpu"),
    ("runs/phase2_bossbase/train_colab.log", "runs/phase2_bossbase",  "phase2_bossbase"),
    ("runs/logs_brutos/sweep_lambda.log",    "runs/sweep",            "sweep_lambda"),
]

def extrair_ipynb(caminho):
    """Concatena a saída de texto de todas as células de código de um notebook.

    O Colab guarda a saída em 'text' (stream) ou em data['text/plain'] (execute_result),
    e cada uma pode vir como string ou lista de linhas.
    """
    with open(caminho, encoding="utf-8") as f:
        nb = json.load(f)
    partes = []
    for celula in nb.get("cells", []):
        if celula.get("cell_type") != "code":
            continue
        for saida in celula.get("outputs", []):
            texto = saida.get("text") or (saida.get("data") or {}).get("text/plain") or ""
            partes.append("".join(texto) if isinstance(texto, list) else str(texto))
    return "\n".join(partes)


RE_EPOCA = re.compile(r"\[\s*(treino|val)\s*\]\s+época\s+(\d+)\s+(.*)$")
RE_PAR = re.compile(r"([A-Za-z_][A-Za-z0-9_]*)\s*=\s*([+-]?\d+(?:\.\d+)?)")
RE_BRACO = re.compile(r"=+\s*lambda_image\s*=\s*([\d.]+)\s*=+")

RENOMEAR = {"bit_acc": "acuracia_bits", "W": "w_dist"}


def normalizar(chave):
    return RENOMEAR.get(chave, chave)


def ler_log(caminho):
    """Devolve (linhas_treino, linhas_val). Cada linha é um dict com 'epoca' + métricas."""
    treino, val = [], []
    braco = None
    with open(caminho, encoding="utf-8", errors="replace") as f:
        for linha in f:
            m_braco = RE_BRACO.search(linha)
            if m_braco:
                # a varredura repete as épocas por braço; o lambda vira coluna
                braco = m_braco.group(1)
                continue
            m = RE_EPOCA.search(linha)
            if not m:
                continue
            tipo, epoca, resto = m.group(1), int(m.group(2)), m.group(3)
            registro = {"epoca": epoca}
            if braco is not None:
                registro["lambda_imagem"] = braco
            for chave, valor in RE_PAR.findall(resto):
                # guarda o TEXTO do log, não float(): float("36.30") viraria "36.3" no CSV
                # e deixaria de bater com o número citado no relatório e no ROADMAP.
                registro[normalizar(chave)] = valor
            (treino if tipo == "treino" else val).append(registro)
    return treino, val


def gravar(linhas, destino, ordem_inicial=("epoca", "lambda_imagem")):
    """Grava o CSV com a união das colunas encontradas, sem inventar valor faltante."""
    if not linhas:
        return None
    colunas = [c for c in ordem_inicial if any(c in l for l in linhas)]
    for l in linhas:
        for c in l:
            if c not in colunas:
                colunas.append(c)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    with open(destino, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=colunas, restval="")
        escritor.writeheader()
        escritor.writerows(linhas)
    return destino, len(linhas), colunas


# Runs cujas métricas de validação existem em CSV (geradas aqui ou pelo próprio treino).
RESUMO_FONTES = [
    ("Fase 0 (BOSSBase, GPU)",      "runs/phase0_bossbase/metricas_val.csv"),
    ("Fase 0 (local, CPU, 2k img)", "runs/phase0_local_cpu/metricas_val.csv"),
    ("Fase 2 (WGAN-GP)",            "runs/phase2_bossbase/metricas_val.csv"),
    ("Fase 2.5 (Fase 0, residual)", "runs/phase0_residual/metricas_val.csv"),
    ("Fase 2.5 (Fase 2, residual)", "runs/phase2_residual/metricas_val.csv"),
    ("Fase 3 (SRNet no laço)",      "runs/phase3_bossbase/metricas_val.csv"),
    ("Fase 3 (sanidade, 3 ép.)",    "runs/phase3_sanidade/metricas_val.csv"),
]


def resumir():
    """Uma linha por run: melhor época por PSNR, e a estabilidade nas 10 últimas épocas.

    A média das 10 últimas existe porque "melhor época" esconde pico isolado: a Fase 0
    bateu PSNR >= 35 dB em UMA das 120 épocas. Reportar só o pico seria enganoso.
    """
    linhas = []
    for rotulo, rel in RESUMO_FONTES:
        caminho = os.path.join(RAIZ, rel)
        if not os.path.exists(caminho):
            continue
        with open(caminho, encoding="utf-8") as f:
            dados = [r for r in csv.DictReader(f)]
        if not dados:
            continue
        num = lambda r, k: float(r[k]) if r.get(k) not in (None, "") else None
        com_psnr = [r for r in dados if num(r, "psnr") is not None]
        # Desempate pela ÚLTIMA época, para casar com o ckpt_best.pt: o treino salva
        # a cada melhoria com ">=", então numa sequência empatada fica a mais recente.
        # A Fase 2 empata em 36,30 dB nas épocas 37, 54 e 55 — o checkpoint é o da 55.
        melhor = dados[-1]
        for r in com_psnr:
            if melhor.get("psnr") in (None, "") or num(r, "psnr") >= num(melhor, "psnr"):
                melhor = r
        ult = [r for r in dados[-10:] if num(r, "psnr") is not None]
        media = lambda k, rs: (sum(num(r, k) for r in rs) / len(rs)) if rs else ""
        linhas.append({
            "run": rotulo,
            "epocas": len(dados),
            "melhor_epoca": melhor["epoca"],
            "acuracia_bits": melhor.get("acuracia_bits", ""),
            "psnr": melhor.get("psnr", ""),
            "ssim": melhor.get("ssim", ""),
            "auc_srnet": melhor.get("auc_srnet", ""),
            "psnr_medio_10_ultimas": round(media("psnr", ult), 2) if ult else "",
            "epocas_psnr_maior_igual_35": sum(1 for r in com_psnr if num(r, "psnr") >= 35),
            "fonte": rel,
        })
    return linhas


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--listar", action="store_true", help="não grava, só mostra")
    args = ap.parse_args()

    for nb_rel, log_rel in NOTEBOOKS:
        nb = os.path.join(RAIZ, nb_rel)
        if not os.path.exists(nb):
            print(f"[pular] {nb_rel} não existe")
            continue
        texto = extrair_ipynb(nb)
        if not texto.strip():
            print(f"[pular] {nb_rel} está sem outputs salvos")
            continue
        destino = os.path.join(RAIZ, log_rel)
        if args.listar:
            print(f"[ipynb] {nb_rel} -> {log_rel} ({len(texto.splitlines())} linhas)")
            continue
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        with open(destino, "w", encoding="utf-8") as f:
            f.write(texto)
        print(f"[ipynb] {nb_rel} -> {log_rel} ({len(texto.splitlines())} linhas)")

    for log_rel, saida_rel, rotulo in FONTES:
        log = os.path.join(RAIZ, log_rel)
        if not os.path.exists(log):
            print(f"[pular] {log_rel} não existe")
            continue
        treino, val = ler_log(log)
        print(f"\n### {rotulo}  ({log_rel})")
        for linhas, nome in ((treino, "metricas_treino.csv"), (val, "metricas_val.csv")):
            if not linhas:
                print(f"    {nome:<22} nenhuma linha encontrada")
                continue
            destino = os.path.join(RAIZ, saida_rel, nome)
            if args.listar:
                print(f"    {nome:<22} {len(linhas):>4} linhas  colunas={sorted(linhas[0])}")
                continue
            _, n, colunas = gravar(linhas, destino)
            print(f"    {nome:<22} {n:>4} linhas -> {os.path.relpath(destino, RAIZ)}")
            print(f"    {'':<22}      colunas: {', '.join(colunas)}")

    linhas = resumir()
    if linhas and not args.listar:
        destino = os.path.join(RAIZ, "runs/resumo_fases.csv")
        gravar(linhas, destino, ordem_inicial=("run", "epocas", "melhor_epoca"))
        print(f"\n### resumo consolidado -> {os.path.relpath(destino, RAIZ)} ({len(linhas)} runs)")
        for l in linhas:
            print(f"    {l['run']:<28} ép {l['melhor_epoca']:>3}/{l['epocas']:<3} "
                  f"psnr {l['psnr']:>5}  (méd 10 últimas {l['psnr_medio_10_ultimas']})  "
                  f"acc {l['acuracia_bits']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fase25_residual.py

import os
import shutil
import sys
import time

import torch
import torch.nn as nn

import fase0_codificador_decodificador as base
import fase2_wgan_gp as fase2


class BlocoResidual(nn.Module):

    def __init__(self, canais):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(canais, canais, 3, padding=1),
            nn.BatchNorm2d(canais),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(canais, canais, 3, padding=1),
            nn.BatchNorm2d(canais),
        )

    def forward(self, x):
        return x + self.conv(x)


class CodificadorResidual(nn.Module):
  

    def __init__(self, canais, profundidade_dados, largura=32, blocos=3):
        super().__init__()
        self.features = base.bloco_conv(canais, largura)
        self.entrada = base.bloco_conv(largura + profundidade_dados, largura)
        self.residuais = nn.Sequential(*[BlocoResidual(largura) for _ in range(blocos)])
        self.para_residuo = nn.Conv2d(largura, canais, 3, padding=1)

    def forward(self, portadora, mensagem):
        x = self.features(portadora)
        x = self.entrada(torch.cat([x, mensagem], dim=1))
        x = self.residuais(x)
        residuo = self.para_residuo(x)
        return (portadora + residuo).clamp(0.0, 1.0)


class DecodificadorResidual(nn.Module):

    def __init__(self, canais, profundidade_dados, largura=32, blocos=3):
        super().__init__()
        self.entrada = base.bloco_conv(canais, largura)
        self.residuais = nn.Sequential(*[BlocoResidual(largura) for _ in range(blocos)])
        self.para_logits = nn.Conv2d(largura, profundidade_dados, 3, padding=1)

    def forward(self, estego):
        x = self.entrada(estego)
        x = self.residuais(x)
        return self.para_logits(x)


_ORIGINAIS = {"Codificador": base.Codificador, "Decodificador": base.Decodificador}
_ATUAL = "denso"

BLOCOS_RESIDUAIS = 3   # o STEGS2GAN usa 3


def usar(arquitetura):

    global _ATUAL
    if arquitetura not in ("denso", "residual"):
        raise ValueError('arquitetura deve ser "denso" ou "residual"')

    if arquitetura == "denso":
        base.Codificador = _ORIGINAIS["Codificador"]
        base.Decodificador = _ORIGINAIS["Decodificador"]
    else:
        def _cod(canais, profundidade, largura=32):
            return CodificadorResidual(canais, profundidade, largura, BLOCOS_RESIDUAIS)

        def _dec(canais, profundidade, largura=32):
            return DecodificadorResidual(canais, profundidade, largura, BLOCOS_RESIDUAIS)

        base.Codificador = _cod
        base.Decodificador = _dec

    _ATUAL = arquitetura
    print(f"[arquitetura] gerador = {arquitetura}")
    return arquitetura


def contar_parametros(canais=1, profundidade=1, largura=64):
 
    resultado = {}
    for nome, (cod, dec) in {
        "denso": (_ORIGINAIS["Codificador"], _ORIGINAIS["Decodificador"]),
        "residual": (
            lambda c, p, l: CodificadorResidual(c, p, l, BLOCOS_RESIDUAIS),
            lambda c, p, l: DecodificadorResidual(c, p, l, BLOCOS_RESIDUAIS),
        ),
    }.items():
        c = cod(canais, profundidade, largura)
        d = dec(canais, profundidade, largura)
        resultado[nome] = {
            "codificador": sum(p.numel() for p in c.parameters()),
            "decodificador": sum(p.numel() for p in d.parameters()),
        }
        resultado[nome]["total"] = (resultado[nome]["codificador"]
                                    + resultado[nome]["decodificador"])
    return resultado


class _Tee:

    def __init__(self, caminho):
        self.caminho = caminho
        self.arquivo = None
        self.stdout = None

    def __enter__(self):
        os.makedirs(os.path.dirname(self.caminho) or ".", exist_ok=True)
        self.arquivo = open(self.caminho, "a", encoding="utf-8")
        self.stdout = sys.stdout
        sys.stdout = self
        return self

    def write(self, texto):
        self.stdout.write(texto)
        self.arquivo.write(texto)
        self.arquivo.flush()

    def flush(self):
        self.stdout.flush()
        self.arquivo.flush()

    def __exit__(self, *exc):
        sys.stdout = self.stdout
        self.arquivo.close()


def _gerar_csvs(log, pasta):
    try:
        from logs_para_csv import ler_log, gravar
    except ImportError:
        print("[csv] logs_para_csv.py não encontrado; CSVs não gerados")
        return
    treino, val = ler_log(log)
    for linhas, nome in ((treino, "metricas_treino.csv"), (val, "metricas_val.csv")):
        if linhas:
            gravar(linhas, os.path.join(pasta, nome))
            print(f"[csv] {nome}: {len(linhas)} linhas")


def espelhar(pasta, destino):
    if not destino or os.path.abspath(destino) == os.path.abspath(pasta):
        return
    os.makedirs(destino, exist_ok=True)
    copiados = 0
    for nome in os.listdir(pasta):
        origem = os.path.join(pasta, nome)
        if os.path.isfile(origem):
            shutil.copy2(origem, os.path.join(destino, nome))
            copiados += 1
    print(f"[espelho] {copiados} arquivo(s) copiado(s) para {destino}")


def _rodar(modulo, cfg_extra, pasta_saida, espelho, rotulo):
    rotulo = f"{rotulo} ({_ATUAL})"     # o braço denso também passa por aqui, na sanidade
    modulo.CONFIG.update(base.COLAB_GPU)
    modulo.CONFIG.update(cfg_extra)
    modulo.CONFIG["pasta_saida"] = pasta_saida
    os.makedirs(pasta_saida, exist_ok=True)
    log = os.path.join(pasta_saida, "treino.log")

    inicio = time.time()
    with _Tee(log):
        print(f"\n===== {rotulo} iniciado em "
              f"{time.strftime('%Y-%m-%d %H:%M:%S')} =====")
        print(f"[arquitetura] {_ATUAL}  |  blocos residuais = {BLOCOS_RESIDUAIS}")
        print(f"[cfg] " + "  ".join(
            f"{k}={modulo.CONFIG[k]}" for k in
            ("tamanho_img", "tamanho_msg", "tamanho_lote", "largura", "epocas",
             "lambda_imagem", "lambda_residuo") if k in modulo.CONFIG))
        resultado = modulo.treinar()
        minutos = (time.time() - inicio) / 60
        print(f"[tempo ] {rotulo}: {minutos:.1f} min "
              f"({minutos * 60 / max(modulo.CONFIG['epocas'], 1):.1f} s/época)")

    _gerar_csvs(log, pasta_saida)
    espelhar(pasta_saida, espelho)
    return resultado


def treinar_fase0(pasta_saida, espelho=None, **cfg_extra):
    return _rodar(base, cfg_extra, pasta_saida, espelho, "Fase 0")


def treinar_fase2(pasta_saida, espelho=None, **cfg_extra):
    return _rodar(fase2, cfg_extra, pasta_saida, espelho, "Fase 2")


ALVO_ACURACIA = 0.99
ALVO_PSNR_FASE2 = 36.30      # o que a Fase 2 densa entregou; não pode regredir


def _ler_csv(caminho):
    import csv
    if not os.path.exists(caminho):
        return []
    with open(caminho, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def comparar(csv_denso, csv_residual, alvo_psnr=ALVO_PSNR_FASE2, rotulo="Fase 2"):
 
    d, r = _ler_csv(csv_denso), _ler_csv(csv_residual)
    if not r:
        print(f"[comparar] {csv_residual} não encontrado — rode o treino antes.")
        return None
    if not d:
        print(f"[comparar] {csv_denso} não encontrado — sem linha de base para comparar.")
        return None

    n = min(len(d), len(r))
    print(f"\n{'=' * 74}\n{rotulo}: DENSO × RESIDUAL nas primeiras {n} épocas\n{'=' * 74}")

    def resumo(linhas, nome):
        linhas = linhas[:n]
        psnr = [float(x["psnr"]) for x in linhas if x.get("psnr")]
        acc = [float(x["acuracia_bits"]) for x in linhas if x.get("acuracia_bits")]
        ssim = [float(x["ssim"]) for x in linhas if x.get("ssim")]
        melhor = max(psnr) if psnr else 0.0
        ult10 = psnr[-10:] or [0.0]
        print(f"  {nome:<9} melhor PSNR {melhor:6.2f} dB | "
              f"média 10 últimas {sum(ult10) / len(ult10):6.2f} dB | "
              f"acurácia mín {min(acc) if acc else 0:.4f} | "
              f"SSIM máx {max(ssim) if ssim else 0:.4f} | "
              f"épocas >= {alvo_psnr:.2f} dB: {sum(1 for p in psnr if p >= alvo_psnr)}")
        return {"melhor": melhor, "acc_min": min(acc) if acc else 0.0,
                "media10": sum(ult10) / len(ult10)}

    rd, rr = resumo(d, "denso"), resumo(r, "residual")

    print(f"\n  diferença (residual - denso): "
          f"{rr['melhor'] - rd['melhor']:+.2f} dB no melhor, "
          f"{rr['media10'] - rd['media10']:+.2f} dB na média das 10 últimas")

    print(f"\n{'-' * 74}\nCRITÉRIOS DE SAÍDA DA FASE 2.5")
    c1 = rr["acc_min"] >= ALVO_ACURACIA
    c2 = rr["melhor"] >= alvo_psnr
    print(f"  (1) acurácia de bits >= {ALVO_ACURACIA}      -> {rr['acc_min']:.4f}  "
          f"{'OK' if c1 else 'FALHOU'}")
    print(f"  (2) PSNR >= {alvo_psnr:.2f} dB (não regredir) -> {rr['melhor']:.2f}  "
          f"{'OK' if c2 else 'FALHOU'}")
    print("  (3) custo por época medido            -> ver a linha [tempo ] em cada treino.log")
    print("  (4) os dois geradores preservados     -> confira o espelho no Drive")

    if c1 and c2:
        print("\n  Residual passa. Decisão de qual segue: maior PSNR a igual acurácia;")
        print("  empate técnico (< 0,2 dB) fica com o mais barato por época.")
    else:
        print("\n  Residual NÃO bateu os critérios. Resultado negativo também é resultado:")
        print("  registre no ROADMAP e siga com o denso.")
    return {"denso": rd, "residual": rr}


### Linha de base do gerador denso

Os CSVs abaixo são os resultados **já medidos** do gerador denso (Fase 0 com 120 épocas e
Fase 2 com 60). A seção 8 compara contra eles. Ficam gravados aqui para o notebook não
depender de o Drive estar organizado de um jeito específico.

In [ ]:
%%writefile denso_fase0_metricas_val.csv
epoca,acuracia_bits,psnr,ssim,linf
1,0.9926,16.70,0.2763,0.7795
2,0.9995,19.83,0.3919,0.7905
3,0.9981,19.83,0.3537,0.7202
4,0.9998,24.66,0.5914,0.6788
5,0.9997,25.45,0.6482,0.5697
6,0.9996,25.19,0.6337,0.5965
7,0.9998,27.26,0.7160,0.4988
8,0.9999,27.08,0.7167,0.5117
9,0.9996,16.62,0.4996,0.8308
10,0.9998,25.57,0.6177,0.5173
11,0.9998,28.91,0.7933,0.4242
12,0.9999,23.37,0.6646,0.5093
13,0.9998,24.75,0.6284,0.4834
14,0.9996,27.11,0.7139,0.4806
15,0.9998,29.57,0.8179,0.4370
16,0.9997,27.01,0.6932,0.4538
17,0.9999,29.37,0.8221,0.4380
18,0.9998,27.62,0.7266,0.4714
19,0.9999,28.04,0.7498,0.5112
20,0.9998,29.96,0.8445,0.4007
21,0.9999,30.30,0.8429,0.4167
22,0.9999,27.05,0.7052,0.4580
23,1.0000,27.58,0.8083,0.4056
24,0.9999,27.99,0.7464,0.4377
25,0.9999,28.88,0.7937,0.4543
26,0.9999,29.22,0.7977,0.4524
27,0.9997,27.79,0.7614,0.4767
28,0.9999,29.28,0.8433,0.4122
29,0.9999,30.63,0.8643,0.4045
30,0.9995,26.59,0.7100,0.5889
31,0.9999,31.13,0.8762,0.3623
32,0.9999,30.39,0.8656,0.4627
33,0.9999,30.04,0.8390,0.5058
34,1.0000,30.95,0.8735,0.4210
35,1.0000,26.29,0.8648,0.4258
36,0.9999,29.46,0.8059,0.4485
37,0.9999,31.35,0.8828,0.3858
38,0.9999,29.77,0.8219,0.4207
39,0.9999,30.67,0.8637,0.4304
40,0.9999,31.10,0.8789,0.4417
41,0.9999,30.25,0.8420,0.4035
42,0.9999,31.28,0.8847,0.3862
43,0.9998,31.74,0.8929,0.3658
44,1.0000,27.39,0.8482,0.4033
45,1.0000,31.71,0.8993,0.3776
46,0.9999,30.37,0.8735,0.4972
47,0.9999,32.00,0.8995,0.3673
48,0.9999,32.11,0.9131,0.4022
49,0.9999,32.56,0.9210,0.3922
50,1.0000,28.32,0.8365,0.3764
51,1.0000,31.64,0.9006,0.4693
52,1.0000,32.22,0.9129,0.3681
53,0.9999,32.93,0.9301,0.3791
54,0.9999,31.05,0.8553,0.4709
55,0.9999,32.56,0.9054,0.4127
56,0.9999,32.63,0.9185,0.4266
57,0.9999,33.03,0.9248,0.3913
58,1.0000,33.09,0.9280,0.3515
59,0.9998,30.52,0.8509,0.4498
60,0.9999,32.36,0.9143,0.4008
61,0.9999,30.47,0.8387,0.5119
62,0.9999,33.18,0.9225,0.3716
63,0.9999,33.44,0.9336,0.3658
64,1.0000,32.89,0.9269,0.3869
65,0.9999,30.95,0.8700,0.4185
66,0.9999,33.07,0.9397,0.3625
67,1.0000,33.83,0.9436,0.3643
68,1.0000,33.40,0.9377,0.3771
69,1.0000,33.42,0.9378,0.3906
70,0.9999,31.59,0.8706,0.4182
71,0.9999,32.77,0.9076,0.3888
72,1.0000,32.49,0.9098,0.4480
73,1.0000,31.81,0.8975,0.3716
74,1.0000,32.90,0.9209,0.4237
75,1.0000,33.61,0.9389,0.3840
76,1.0000,33.00,0.9215,0.4056
77,1.0000,33.64,0.9384,0.3778
78,1.0000,33.89,0.9444,0.3799
79,1.0000,33.47,0.9259,0.3737
80,1.0000,33.15,0.9328,0.3711
81,1.0000,33.92,0.9449,0.3958
82,0.9999,34.43,0.9536,0.3146
83,1.0000,34.11,0.9456,0.3794
84,1.0000,34.13,0.9491,0.3912
85,1.0000,34.18,0.9513,0.3542
86,1.0000,33.62,0.9391,0.3700
87,1.0000,33.45,0.9341,0.3934
88,1.0000,34.21,0.9472,0.3559
89,1.0000,34.08,0.9482,0.3691
90,1.0000,33.24,0.9311,0.4006
91,1.0000,34.41,0.9523,0.3738
92,0.9999,21.74,0.7732,0.4881
93,1.0000,33.79,0.9323,0.3730
94,0.9995,33.41,0.9357,0.4469
95,1.0000,33.76,0.9419,0.4012
96,1.0000,34.34,0.9499,0.3404
97,1.0000,34.10,0.9413,0.4144
98,1.0000,34.28,0.9485,0.4181
99,1.0000,34.42,0.9510,0.3959
100,1.0000,25.13,0.7761,0.3796
101,0.9999,34.43,0.9528,0.4060
102,0.9999,34.62,0.9540,0.3502
103,1.0000,32.50,0.9166,0.4391
104,1.0000,34.26,0.9490,0.4152
105,1.0000,32.80,0.9251,0.4692
106,1.0000,34.03,0.9486,0.3880
107,1.0000,34.24,0.9510,0.3788
108,0.9999,34.58,0.9576,0.4488
109,0.9999,35.16,0.9625,0.3622
110,1.0000,34.39,0.9482,0.3415
111,1.0000,34.73,0.9551,0.3537
112,1.0000,34.70,0.9559,0.3819
113,0.9999,29.85,0.8545,0.5065
114,1.0000,34.43,0.9533,0.3573
115,1.0000,33.84,0.9427,0.3880
116,1.0000,34.62,0.9532,0.3498
117,0.9999,33.94,0.9447,0.3929
118,1.0000,34.20,0.9420,0.3766
119,1.0000,33.86,0.9388,0.3895
120,1.0000,34.55,0.9532,0.3947

In [ ]:
%%writefile denso_fase2_metricas_val.csv
epoca,acuracia_bits,psnr,ssim,linf
1,0.9999,34.80,0.9632,0.3725
2,0.9998,35.34,0.9667,0.3667
3,0.9998,34.94,0.9626,0.3425
4,0.9998,35.47,0.9675,0.3528
5,0.9998,35.41,0.9684,0.3728
6,0.9998,35.36,0.9683,0.3820
7,0.9998,35.57,0.9690,0.4084
8,0.9998,35.49,0.9671,0.3438
9,0.9997,35.39,0.9662,0.3556
10,0.9997,35.68,0.9705,0.3304
11,0.9997,35.67,0.9704,0.3226
12,0.9997,35.73,0.9708,0.3493
13,0.9998,35.61,0.9694,0.3668
14,0.9997,35.78,0.9708,0.3620
15,0.9997,35.81,0.9706,0.3663
16,0.9998,35.62,0.9691,0.3313
17,0.9997,35.66,0.9696,0.3330
18,0.9988,33.87,0.9602,0.3788
19,0.9998,35.34,0.9658,0.3719
20,0.9998,35.88,0.9716,0.3310
21,0.9997,35.85,0.9708,0.3090
22,0.9998,35.64,0.9698,0.3589
23,0.9998,35.53,0.9656,0.3450
24,0.9997,34.91,0.9653,0.3697
25,0.9997,35.89,0.9716,0.3880
26,0.9998,36.01,0.9724,0.3599
27,0.9997,36.07,0.9726,0.4230
28,0.9998,35.89,0.9721,0.4284
29,0.9998,35.99,0.9712,0.3755
30,0.9997,36.02,0.9724,0.4237
31,0.9997,35.97,0.9720,0.3975
32,0.9997,36.24,0.9728,0.4074
33,0.9998,35.83,0.9710,0.3837
34,0.9997,36.09,0.9731,0.4274
35,0.9997,36.05,0.9731,0.3366
36,0.9998,35.77,0.9702,0.3706
37,0.9997,36.30,0.9731,0.3878
38,0.9998,35.63,0.9703,0.3570
39,0.9997,36.20,0.9731,0.3867
40,0.9998,35.55,0.9706,0.4385
41,0.9997,36.10,0.9742,0.4108
42,0.9998,36.00,0.9721,0.3706
43,0.9997,36.11,0.9733,0.3591
44,0.9997,36.12,0.9737,0.3871
45,0.9998,36.21,0.9736,0.3584
46,0.9997,35.98,0.9711,0.3595
47,0.9998,36.03,0.9730,0.3994
48,0.9997,36.10,0.9739,0.3798
49,0.9998,35.85,0.9690,0.3834
50,0.9998,36.03,0.9721,0.3343
51,0.9998,35.97,0.9728,0.3912
52,0.9997,36.22,0.9735,0.3476
53,0.9996,36.26,0.9750,0.4022
54,0.9997,36.30,0.9746,0.3785
55,0.9997,36.30,0.9743,0.3596
56,0.9997,36.10,0.9723,0.3808
57,0.9998,35.83,0.9711,0.4048
58,0.9997,36.21,0.9742,0.3907
59,0.9997,36.06,0.9738,0.4486
60,0.9997,36.21,0.9737,0.3781

## 4. Dataset (BOSSBase, grayscale)

Espera `datasets/BOSSbase_1.01` dentro de `PASTA_BASE`. Ler direto do Drive é lento; se a
sessão aguentar, copiar para o disco local acelera bastante o treino.

In [ ]:
import os, glob, shutil, time

ORIGEM = os.path.join(PASTA_DADOS, 'BOSSbase_1.01')
LOCAL  = '/content/datasets/BOSSbase_1.01'
COPIAR_PARA_LOCAL = True

assert os.path.isdir(ORIGEM), f'não encontrei {ORIGEM}'
n_origem = len(glob.glob(os.path.join(ORIGEM, '*.pgm')))
print(f'{n_origem} imagens em {ORIGEM}')

if COPIAR_PARA_LOCAL:
    if not os.path.isdir(LOCAL) or len(glob.glob(LOCAL + '/*.pgm')) < n_origem:
        t0 = time.time()
        shutil.copytree(ORIGEM, LOCAL, dirs_exist_ok=True)
        print(f'copiado para {LOCAL} em {(time.time()-t0)/60:.1f} min')
    PASTA_BOSS = LOCAL
else:
    PASTA_BOSS = ORIGEM
print('usando:', PASTA_BOSS, '->', len(glob.glob(PASTA_BOSS + '/*.pgm')), 'imagens')

## 5. Orçamento: parâmetros e segundos por época

Roda 2 épocas com cada arquitetura para medir o custo real **antes** de comprometer a noite.
Isto já entrega o critério de saída (3).

Duas coisas a observar no resultado:

- **as duas arquiteturas não têm o mesmo tamanho** para a mesma `largura`. O roadmap manda
  variar uma coisa por vez, e a variável desta fase é a topologia a largura fixa — então a
  diferença de parâmetros é *parte* do que está sendo comparado, não um erro. Se depois
  quiser separar topologia de capacidade, rode um terceiro braço com a largura do residual
  reduzida até igualar o total do denso;
- o **ETA** impresso no fim decide entre o protocolo fiel (120 + 60 épocas, igual ao denso)
  e o curto (60 + 40), caso a noite não comporte.

In [ ]:
import importlib, time
import fase0_codificador_decodificador as base, fase2_wgan_gp as fase2, fase25_residual as f25
for m in (base, fase2, f25):
    importlib.reload(m)

par = f25.contar_parametros(canais=1, profundidade=1, largura=64)
print('PARÂMETROS (largura=64)')
for nome, v in par.items():
    print(f"  {nome:<9} codificador {v['codificador']:>8,} | decodificador {v['decodificador']:>8,}"
          f" | total {v['total']:>9,}")
print(f"  razão residual/denso: {par['residual']['total']/par['denso']['total']:.2f}x\n")

CFG_SANIDADE = dict(pasta_dados=PASTA_BOSS, canais=1, max_imagens=1000,
                    epocas=2, num_trabalhadores=2, salvar_cada=99)

tempos = {}
for arq in ('denso', 'residual'):
    f25.usar(arq)
    t0 = time.time()
    f25.treinar_fase0(pasta_saida=f'{TRABALHO}/sanidade_{arq}', **CFG_SANIDADE)
    tempos[arq] = (time.time() - t0) / CFG_SANIDADE['epocas']

print('\n' + '=' * 70)
fator = 10000 / CFG_SANIDADE['max_imagens']
for arq, s in tempos.items():
    print(f'  {arq:<9} {s:6.1f} s/época com {CFG_SANIDADE["max_imagens"]} imagens'
          f'  ->  ~{s*fator:6.1f} s/época com o dataset inteiro')
s_res = tempos['residual'] * fator
print(f"\n  ETA residual (Fase 0 sem crítico ~{s_res:.0f} s/ép; Fase 2 com crítico ~{s_res*3:.0f} s/ép):")
for e0, e2, nome in ((120, 60, 'protocolo fiel (igual ao denso)'), (60, 40, 'protocolo curto')):
    h = (e0 * s_res + e2 * s_res * 3) / 3600
    print(f'    {nome:<34} {e0} + {e2} épocas  ->  ~{h:.1f} h')
print('\n  A Fase 2 é ~3x mais cara por época: n_critic=5 significa 5 passos de crítico'
      '\n  por passo do gerador. O multiplicador acima é estimativa; confira na prática.')

## 6. Fase 0 com o gerador residual

**Não há warm start aqui** — arquitetura diferente, `state_dict` incompatível. Treina do zero.

Se a sessão cair no meio, esta fase **recomeça do zero**: `phase0` não tem retomada. É o
motivo de o protocolo curto existir. O `espelho` copia tudo para o Drive a cada término, e
o `ckpt_best.pt` é gravado a cada melhoria dentro da própria pasta de trabalho.

In [ ]:
import importlib
import fase0_codificador_decodificador as base, fase2_wgan_gp as fase2, fase25_residual as f25
for m in (base, fase2, f25):
    importlib.reload(m)

EPOCAS_FASE0 = 120
SAIDA_F0 = f'{TRABALHO}/phase0_residual'

f25.usar('residual')
resultado_f0 = f25.treinar_fase0(
    pasta_saida     = SAIDA_F0,
    espelho         = os.path.join(PASTA_RUNS, 'phase0_residual'),
    pasta_dados     = PASTA_BOSS,
    canais          = 1,
    max_imagens     = None,
    epocas          = EPOCAS_FASE0,
    num_trabalhadores = 4,
)
print(resultado_f0)

## 7. Fase 2 com o gerador residual (WGAN-GP)

Warm start da Fase 0 **residual** — nunca do checkpoint denso. Tudo o mais fica igual ao run
denso: `lambda_adv=0.01`, `n_critic=5`, mesma semente, mesmo payload. Uma variável por vez.

In [ ]:
EPOCAS_FASE2 = 60
SAIDA_F2 = f'{TRABALHO}/phase2_residual'

f25.usar('residual')
resultado_f2 = f25.treinar_fase2(
    pasta_saida     = SAIDA_F2,
    espelho         = os.path.join(PASTA_RUNS, 'phase2_residual'),
    carregar_fase0  = os.path.join(SAIDA_F0, 'ckpt_best.pt'),
    pasta_dados     = PASTA_BOSS,
    canais          = 1,
    max_imagens     = None,
    epocas          = EPOCAS_FASE2,
    num_trabalhadores = 4,
)
print(resultado_f2)

## 8. Comparação denso × residual e veredito

A comparação trunca o mais longo dos dois: se o residual rodou 60 épocas e o denso 120, ela
usa as 60 primeiras dos dois lados. Comparar o melhor de 120 contra o melhor de 60 não seria
comparação.

In [ ]:
f25.comparar('denso_fase0_metricas_val.csv',
             os.path.join(SAIDA_F0, 'metricas_val.csv'),
             alvo_psnr=35.16, rotulo='Fase 0')

f25.comparar('denso_fase2_metricas_val.csv',
             os.path.join(SAIDA_F2, 'metricas_val.csv'),
             alvo_psnr=36.30, rotulo='Fase 2')

## 9. Amostras (portadora / estego / resíduo)

In [ ]:
import glob
from IPython.display import Image, display
for saida, titulo in ((SAIDA_F0, 'Fase 0 residual'), (SAIDA_F2, 'Fase 2 residual')):
    amostras = sorted(glob.glob(os.path.join(saida, 'samples_epoch*.png')))
    if amostras:
        print(f'{titulo} — última amostra: {os.path.basename(amostras[-1])}')
        display(Image(amostras[-1]))

## 10. Antes de fechar a sessão

Confere o que sobreviveu no Drive. A linha que importa é a dos `.pt`: **curva sem peso é
resultado que não se pode continuar**, e o critério de saída (4) exige os dois geradores
preservados.

Depois de rodar, salve o notebook **com os outputs**. O `treino.log` e os CSVs já cobrem o
registro, mas os notebooks anteriores deste projeto foram salvos sem output e o log da Fase 2
só foi recuperado por acaso.

In [ ]:
import os
print('CONTEÚDO ESPELHADO NO DRIVE\n' + '=' * 60)
for sub in ('phase0_residual', 'phase2_residual'):
    pasta = os.path.join(PASTA_RUNS, sub)
    if not os.path.isdir(pasta):
        print(f'  {sub}: NÃO EXISTE  <-- espelho falhou, copie à mão antes de fechar')
        continue
    arquivos = sorted(os.listdir(pasta))
    pesos = [a for a in arquivos if a.endswith('.pt')]
    registros = [a for a in arquivos if a.endswith(('.log', '.csv'))]
    print(f'  {sub}: {len(arquivos)} arquivos')
    print(f'     pesos     : {pesos if pesos else "*** NENHUM ***"}')
    print(f'     registros : {registros if registros else "*** NENHUM ***"}')

print('\nSe algum "pesos" ou "registros" estiver vazio, rode:')
print("  f25.espelhar(SAIDA_F0, os.path.join(PASTA_RUNS, 'phase0_residual'))")
print("  f25.espelhar(SAIDA_F2, os.path.join(PASTA_RUNS, 'phase2_residual'))")